In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.optimizers import Adam
import spacy
from sklearn.model_selection import train_test_split

## **Loading and viewing the dataset**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/slogan-valid.csv")
df = data[["industry","output"]]
df = df.dropna()
df.head()

,industry,output
0,computer hardware,Taking Care of Small Business Technology
1,"health, wellness and fitness",Build World-Class Recreation Programs
2,internet,Most Powerful Lead Generation Software for Mar...
3,internet,Hire quality freelancers for your job
4,financial services,"Financial Advisers Norwich, Norfolk"


## **Data Preprocessing**

In [4]:
# Load spaCy model for text processing
nlp = spacy.load("en_core_web_sm")

def preprocess_text(text):
    text_lower = text.lower()
    doc = nlp(text_lower)

    processed_tokens = []

    for token in doc:
        if not token.is_punct:
            processed_tokens.append(token.text)

    return " ".join(processed_tokens)

df["processed_slogan"] = df["output"].apply(preprocess_text)

df.head()

,industry,output,processed_slogan
0,computer hardware,Taking Care of Small Business Technology,taking care of small business technology
1,"health, wellness and fitness",Build World-Class Recreation Programs,build world class recreation programs
2,internet,Most Powerful Lead Generation Software for Mar...,most powerful lead generation software for mar...
3,internet,Hire quality freelancers for your job,hire quality freelancers for your job
4,financial services,"Financial Advisers Norwich, Norfolk",financial advisers norwich norfolk


In [5]:
df["modified_slogan"] = df["industry"] + " " + df["processed_slogan"]
df.head()

,industry,output,processed_slogan,modified_slogan
0,computer hardware,Taking Care of Small Business Technology,taking care of small business technology,computer hardware taking care of small busines...
1,"health, wellness and fitness",Build World-Class Recreation Programs,build world class recreation programs,"health, wellness and fitness build world class..."
2,internet,Most Powerful Lead Generation Software for Mar...,most powerful lead generation software for mar...,internet most powerful lead generation softwar...
3,internet,Hire quality freelancers for your job,hire quality freelancers for your job,internet hire quality freelancers for your job
4,financial services,"Financial Advisers Norwich, Norfolk",financial advisers norwich norfolk,financial services financial advisers norwich ...


In [6]:
# Tokenizer to convert words into numerical values tokens
tokenizer = Tokenizer()

# Tokenizer learns words in dataset
tokenizer.fit_on_texts(df["modified_slogan"])

# Total number of unique words in learned vocabulary
total_words = len(tokenizer.word_index) + 1

# Dictionary mapping words to its numerical index: index based on frequency i.e., more freq => lower index
tokenizer.word_index

# Creating input sequences
# Initialise list to store the input sequences
input_sequences = []

# Iterate over processed slogans
for line in df["modified_slogan"]:

    # Convert slogans to token sequences
    token_list = tokenizer.texts_to_sequences([line])[0] # returns list containing list of words indices; extracting inner list [0]


    # Building list of progressively longer input sequences for better training
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

In [7]:
max_seq_len = max([len(seq) for seq in input_sequences])
max_seq_len

15

## **Training Data for Slogan Generator**

In [8]:
input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding="pre")

In [9]:
X_gen = input_sequences[:, :-1]
y_gen = input_sequences[:, -1]

In [10]:
y_gen = tf.keras.utils.to_categorical(y_gen, num_classes=total_words)

## **Slogan Generator Architecture**

In [11]:
gen_model = Sequential([
    Embedding(total_words, 100, input_length=max_seq_len-1),
    LSTM(150, return_sequences=True),
    LSTM(100),
    Dense(total_words, activation="softmax")
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [12]:
gen_model.compile(
    loss="categorical_crossentropy",
    optimizer=Adam(),
    metrics=["accuracy"]
)

## **Slogan Generation**

In [13]:
gen_model.fit(X_gen, y_gen, epochs=50, verbose=1)

Epoch 1/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 39s 33ms/step - accuracy: 0.0656 - loss: 7.3291
Epoch 2/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 37s 34ms/step - accuracy: 0.0980 - loss: 6.3792
Epoch 3/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 40s 33ms/step - accuracy: 0.1464 - loss: 5.9926
Epoch 4/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 38s 35ms/step - accuracy: 0.1757 - loss: 5.7124
Epoch 5/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 37s 34ms/step - accuracy: 0.1963 - loss: 5.4757
Epoch 6/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 37s 34ms/step - accuracy: 0.2182 - loss: 5.2243
Epoch 7/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 37s 34ms/step - accuracy: 0.2329 - loss: 5.0189
Epoch 8/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 37s 34ms/step - accuracy: 0.2402 - loss: 4.8249
Epoch 9/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 37s 34ms/step - accuracy: 0.2538 - loss: 4.5812
Epoch 10/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 37s 34ms/step - accuracy: 0.2604 - loss: 4.4179
Epoch 11/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 36s 34ms/step - accuracy: 0.2738 - loss: 4.2203
Epoch 12

In [14]:
def generate_slogan(seed_text, max_words=20):
    for _ in range(max_words):

        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding="pre")

        predictions = gen_model.predict(token_list, verbose=0)

        predicted_index = np.argmax(predictions)

        output_word = None

        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break

        if output_word is None:
            break

        seed_text += " " + output_word

    return seed_text

In [15]:
# Slogan generation
generate_slogan("internet")

'internet web design and development company in delhi oh area today area city online management online staffing improve membership calicut text'

## **Training Data for Slogan Classifier**

In [16]:
industries = df["industry"].unique()
industries

array(['computer hardware', 'health, wellness and fitness', 'internet',
       'financial services', 'mechanical or industrial engineering',
       'marketing and advertising', 'hospital & health care', 'research',
       'information technology and services', 'computer software',
       'oil & energy', 'dairy', 'transportation/trucking/railroad',
       'design', 'furniture', 'professional training & coaching',
       'hospitality', 'textiles', 'food & beverages',
       'management consulting', 'medical practice', 'accounting',
       'performing arts', 'electrical/electronic manufacturing',
       'higher education', 'outsourcing/offshoring',
       'venture capital & private equity', 'writing and editing',
       'mining & metals', 'construction', 'consumer electronics',
       'retail', 'human resources', 'staffing and recruiting', 'farming',
       'wholesale', 'events services', 'import and export',
       'non-profit organization management', 'machinery',
       'information se

In [17]:
industry_to_index = {industry: idx for idx, industry in enumerate(industries)}

In [18]:
df["industry_index"] = df["industry"].map(industry_to_index)

In [21]:
industry_counts = df["industry"].value_counts()
industry_counts.sort_values().head(20)

,count
industry,
public policy,1
supermarkets,1
military,1
tobacco,1
libraries,1
judiciary,1
political organization,2
international affairs,2
package/freight delivery,2


In [22]:
# Removing rare industries before splitting
min_per_class = 2
valid_industries = df["industry"].value_counts()
valid_industries = valid_industries[valid_industries >= min_per_class].index

df = df[df["industry"].isin(valid_industries)].copy()

industries = df["industry"].unique()
industry_to_index = {industry:idx for idx, industry in enumerate(industries)}
df["industry_index"] = df["industry"].map(industry_to_index)

In [23]:
df_train, df_test = train_test_split(
    df,
    test_size=0.2,
    stratify=df["industry_index"],
    random_state=42
)

In [24]:
X_train = tokenizer.texts_to_sequences(df_train["processed_slogan"])
X_test = tokenizer.texts_to_sequences(df_test["processed_slogan"])

In [25]:
X_train = pad_sequences(X_train, maxlen=max_seq_len, padding="pre")
X_test = pad_sequences(X_test, maxlen=max_seq_len, padding="pre")

In [26]:
y_train = tf.keras.utils.to_categorical(
    df_train["industry_index"],
    num_classes=len(industries)
)

y_test = tf.keras.utils.to_categorical(
    df_test["industry_index"],
    num_classes=len(industries)
)

## **Slogan Classifier Architecture**

In [27]:
class_model = Sequential([
    Embedding(total_words, 100, input_length=max_seq_len),
    LSTM(150, return_sequences=True),
    LSTM(100),
    Dense(len(industries), activation="softmax")
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


## **Slogan Classification & Evaluation**

In [28]:
class_model.compile(
    loss="categorical_crossentropy",
    optimizer=Adam(),
    metrics=["accuracy"]
)

In [29]:
class_model.fit(X_train, y_train, epochs=50, verbose=1)

Epoch 1/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - accuracy: 0.0813 - loss: 4.5035
Epoch 2/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 5s 36ms/step - accuracy: 0.0854 - loss: 4.2842
Epoch 3/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 4s 28ms/step - accuracy: 0.0881 - loss: 4.2616
Epoch 4/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 4s 28ms/step - accuracy: 0.1214 - loss: 3.9868
Epoch 5/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 5s 34ms/step - accuracy: 0.1960 - loss: 3.5797
Epoch 6/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.2500 - loss: 3.2239
Epoch 7/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 6s 36ms/step - accuracy: 0.3049 - loss: 2.9122
Epoch 8/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step - accuracy: 0.3730 - loss: 2.6155
Epoch 9/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.4180 - loss: 2.3646
Epoch 10/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 6s 35ms/step - accuracy: 0.5040 - loss: 2.0915
Epoch 11/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 4s 28ms/step - accuracy: 0.5476 - loss: 1.8885
Epoch 12/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 

In [31]:
loss, accuracy = class_model.evaluate(X_test, y_test)
print("Test Accuracy:", accuracy)

34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.1604 - loss: 7.5614
Test Accuracy: 0.17977528274059296


In [35]:
majority_acc = df_test["industry_index"].value_counts(normalize=True).max()
print("Majority-class baseline accuracy:", majority_acc)
print("Model accuracy:", accuracy)

Majority-class baseline accuracy: 0.08426966292134831
Model accuracy: 0.17977528274059296


The classifier achieved approximately 18% accuracy, which is more than double the majority-class baseline of 8.4%, indicating that the model has learned meaningful industry-related patterns.

In [36]:
def classify_slogan(slogan):

    slogan = preprocess_text(slogan)

    sequence = tokenizer.texts_to_sequences([slogan])

    padded_sequence = pad_sequences(sequence, maxlen=max_seq_len, padding="pre")

    prediction = class_model.predict(padded_sequence, verbose=0)

    predicted_index = np.argmax(prediction)

    return industries[predicted_index]

In [37]:
industry = "internet"
generated_slogan = generate_slogan(industry)
predicted_industry = classify_slogan(generated_slogan)

print("Generated Slogan:", generated_slogan)
print("Predicted Industry:", predicted_industry)

Generated Slogan: internet web design and development company in delhi oh area today area city online management online staffing improve membership calicut text
Predicted Industry: apparel & fashion


### Interpretation
The classifier achieved approximately 18% accuracy, which is more than double the majority-class baseline of 8.4%, indicating that the model has learned meaningful industry-related patterns. However, performance remains modest due to semantic overlap between industries and the short length of slogans. When combining both models, the generator produced industry-related keywords but also included repetitive and location-specific tokens, leading to occasional misclassification. This suggests the generator has learned structural language patterns but lacks strong semantic control over industry-specific vocabulary.